In [ ]:
import pandas as pd
from pathlib import Path
import random
import numpy as np
import cv2
from tqdm.auto import tqdm
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold


c:\Users\Toranj\Desktop\AIM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# clip extraction 
def extract_pre_crash_clip(video_path , crash_time , output_path=None , clip_seconds=4.0 , warning_seconds=1.0 , num_frames=16 , image_size=(224, 224) , short_clip="pad"):

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0 or total_frames <= 0:
        cap.release()
        raise ValueError("Invalid video FPS or frame count.")

    duration = total_frames / fps

    end_time = float(crash_time) - warning_seconds
    start_time = end_time - clip_seconds

    if end_time <= 0 or end_time > duration:
        cap.release()
        return None, {
            "status": "skipped",
            "reason": "Not enough footage before the target horizon",
            "duration": duration,
            "start_time": start_time,
            "end_time": end_time,
        }

    if start_time < 0 and short_clip == "skip":
        cap.release()
        return None, {
            "status": "skipped",
            "reason": "Less than 4 seconds of available footage",
            "duration": duration,
            "start_time": start_time,
            "end_time": end_time,
        }

    times = np.linspace(
        start_time,
        end_time,
        num_frames,
        endpoint=False,
    )

    frames = []

    for t in times:
        t = max(0.0, float(t))

        frame_index = min(
            int(t * fps),
            total_frames - 1,
        )

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        success, frame = cap.read()

        if not success:
            cap.release()
            raise ValueError(f"Could not read frame at {t:.2f}s")

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, image_size)
        frames.append(frame)

    cap.release()

    clip = np.stack(frames).astype(np.uint8)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(output_path, clip)

    return clip, {
        "status": "ok",
        "duration": duration,
        "start_time": start_time,
        "end_time": end_time,
        "padded_seconds": max(0.0, -start_time),
        "shape": clip.shape,
    }

In [2]:
df = pd.read_csv('nexar_data/metadata.csv')

In [3]:
# target column 
df["label"] = df["time_of_event"].notna().astype("int64")

In [ ]:

# first version with leakage 

OUTPUT_DIR = Path("nexar_data/processed_frames_randomnegetive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLIP_SECONDS = 4.0
WARNING_SECONDS = 1.0
NUM_FRAMES = 16
IMAGE_SIZE = (224, 224)

rng = random.Random(42)
results = []


def get_duration(p):
    cap = cv2.VideoCapture(str(p))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    if fps <= 0 or n <= 0:
        return None
    return n / fps

pos_rows = df[(df["label"] == 1) & df["time_of_event"].notna()]
pos_rows = pos_rows[pos_rows["time_of_event"].astype(float) >= CLIP_SECONDS + WARNING_SECONDS]

pos_after = []
for r in pos_rows.itertuples():
    d = get_duration(r.video_file)
    if d is None:
        continue
    after = d - float(r.time_of_event)
    if after >= 0:
        pos_after.append(after)

pos_after = np.array(pos_after)
print("Pool size:", len(pos_after),
      "| min/median/max:", pos_after.min().round(2), np.median(pos_after).round(2), pos_after.max().round(2))


for _, row in tqdm(df.iterrows(), total=len(df)):
    video_path = Path(str(row["video_file"]))
    video_id = str(row["video_id"])
    label = int(row["label"])

    record = {
        "video_id": video_id,
        "label": label,
        "status": None,
        "clip_path": None,
        "reason": None,
        "video_duration": None,
        "start_time": None,
        "end_time": None,
        "padded_seconds": None,
    }

    try:
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()

        if fps <= 0 or total_frames <= 0:
            raise ValueError("Invalid FPS or frame count")

        duration = total_frames / fps
        record["video_duration"] = duration

        if duration < CLIP_SECONDS:
            raise ValueError("Video is shorter than 4 seconds")

        # POSITIVE: clip ends 1 second before the crash.
        if label == 1:
            if pd.isna(row["time_of_event"]):
                raise ValueError("Positive sample has no time_of_event")

            event_time = float(row["time_of_event"])
            start_time = event_time - WARNING_SECONDS - CLIP_SECONDS
            end_time = event_time - WARNING_SECONDS

            if start_time < 0 or end_time > duration:
                raise ValueError("Insufficient footage for pre-crash clip")

            target_time = event_time

        # NEGATIVE
        # follows the same distribution as in positive videos.
        else:
            valid = pos_after[pos_after <= duration - (CLIP_SECONDS + WARNING_SECONDS)]
            if len(valid) == 0:
                raise ValueError("No valid pseudo-event time for this video")

            target_time = duration - float(rng.choice(valid))
            start_time = target_time - WARNING_SECONDS - CLIP_SECONDS
            end_time = target_time - WARNING_SECONDS

            if start_time < 0 or end_time > duration:
                raise ValueError("Pseudo-event leaves an invalid clip window")

        clip_path = OUTPUT_DIR / f"{video_id}.npy"

        if label == 1 and clip_path.exists():
            record.update(status="ok_existing", clip_path=str(clip_path),
                          start_time=start_time, end_time=end_time, padded_seconds=0)
            results.append(record)
            continue

        if clip_path.exists():
            clip_path.unlink()

        clip, info = extract_pre_crash_clip(
            video_path=video_path,
            crash_time=target_time,
            output_path=clip_path,
            clip_seconds=CLIP_SECONDS,
            warning_seconds=WARNING_SECONDS,
            num_frames=NUM_FRAMES,
            image_size=IMAGE_SIZE,
            short_clip="skip",
        )

        record["status"] = info["status"]
        record["reason"] = info.get("reason")
        record["start_time"] = info.get("start_time")
        record["end_time"] = info.get("end_time")
        record["padded_seconds"] = info.get("padded_seconds", 0)

        if clip is not None:
            record["clip_path"] = str(clip_path)

    except Exception as e:
        record["status"] = "error"
        record["reason"] = str(e)

    results.append(record)

    if len(results) % 100 == 0:
        pd.DataFrame(results).to_csv(OUTPUT_DIR / "processing_log.csv", index=False)

processed_df = pd.DataFrame(results)
processed_df.to_csv(OUTPUT_DIR / "processing_log.csv", index=False)

good = processed_df["status"].isin(["ok", "ok_existing"])

print("\nProcessing status:")
print(processed_df["status"].value_counts(dropna=False))

print("\nSuccessful clips by class:")
print(processed_df[good].groupby("label").size())

print("\nSkipped/error samples by class:")
print(processed_df[~good].groupby(["label", "status"]).size())

KeyboardInterrupt: 

In [ ]:

# second version fixed leakage 


OUTPUT_DIR = Path("nexar_data/processed_frames_matched3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLIP_SECONDS   = 4.0
WARNING_SECONDS = 1.0
NUM_FRAMES     = 16
IMAGE_SIZE     = (224, 224)
N_BINS         = 10
N_CANDIDATES   = 30    
rng = random.Random(42)
np.random.seed(42)



pos_rows = df[df.label == 1].copy()
pos_rows = pos_rows.dropna(subset=["time_of_event"])
pos_rows["video_duration"] = pos_rows["video_file"].map(get_duration)
pos_rows = pos_rows[pos_rows["video_duration"].notna() & (pos_rows["video_duration"] >= CLIP_SECONDS + WARNING_SECONDS)]
pos_rows = pos_rows[(pos_rows["time_of_event"] - WARNING_SECONDS - CLIP_SECONDS >= 0) & (pos_rows["time_of_event"] - WARNING_SECONDS <= pos_rows["video_duration"])]
pos_start_times = (pos_rows["time_of_event"] - WARNING_SECONDS - CLIP_SECONDS).to_numpy(dtype=float)
pos_time_after = (pos_rows["video_duration"] - (pos_rows["time_of_event"] - WARNING_SECONDS)).to_numpy(dtype=float)




st_edges = np.unique(np.quantile(pos_start_times, np.linspace(0, 1, N_BINS + 1)))
ta_edges = np.unique(np.quantile(pos_time_after,  np.linspace(0, 1, N_BINS + 1)))
st_idx = np.clip(np.digitize(pos_start_times, st_edges[1:-1]), 0, len(st_edges) - 2)
ta_idx = np.clip(np.digitize(pos_time_after,  ta_edges[1:-1]), 0, len(ta_edges) - 2)
pos_cells = np.stack([st_idx, ta_idx], 1)
uniq_cells, pos_cell_counts = np.unique(pos_cells, axis=0, return_counts=True)
pos_cell_target = dict(zip(map(tuple, uniq_cells), pos_cell_counts))



neg_pool = []   
neg_rows = df[df.label == 0]
for _, row in tqdm(neg_rows.iterrows(), total=len(neg_rows), desc="neg candidates"):
    d = get_duration(row.video_file)
    if d is None or d < CLIP_SECONDS:
        continue

    tried = 0
    while tried < N_CANDIDATES:
        tried += 1

        c = rng.choices(range(len(uniq_cells)), weights=pos_cell_counts, k=1)[0]
        s_bin, t_bin = uniq_cells[c]
        lo_s, hi_s = st_edges[s_bin], st_edges[s_bin + 1]
        lo_t, hi_t = ta_edges[t_bin], ta_edges[t_bin + 1]
        start_time = rng.uniform(lo_s, hi_s)
        after      = rng.uniform(lo_t, hi_t)
        target_time = start_time + CLIP_SECONDS + WARNING_SECONDS
        implied_after = d - target_time
        if implied_after < 0:
            continue
        if not (lo_t - 1e-9 <= implied_after <= hi_t + 1e-9):
            continue
        neg_pool.append(dict(video_id=str(row.video_id), row_idx=row.name , target_time=target_time , start_time=start_time , end_time=start_time + CLIP_SECONDS, duration=d , after=implied_after))
neg_pool = pd.DataFrame(neg_pool)
print(f"candidate pool: {len(neg_pool)} over {neg_pool.video_id.nunique()} videos")
if len(neg_pool) == 0:
    raise RuntimeError("Empty candidate pool — loosen cell constraints or raise N_CANDIDATES.")


neg_pool["s_bin"] = np.clip(np.digitize(neg_pool.start_time, st_edges[1:-1]), 0, len(st_edges)-2)
neg_pool["t_bin"] = np.clip(np.digitize(neg_pool.after,       ta_edges[1:-1]), 0, len(ta_edges)-2)
neg_pool["cell"]   = list(zip(neg_pool.s_bin, neg_pool.t_bin))


rng2 = random.Random(42)
selected = []
used_videos = set()

for cell, target_n in sorted(pos_cell_target.items(), key=lambda kv: -kv[1]):
    cand = neg_pool[neg_pool.cell == cell]
    cand = cand[~cand.video_id.isin(used_videos)]
    if len(cand) == 0:
        continue
    take = min(target_n, len(cand))
    picked = cand.sample(n=take, random_state=rng2.randint(0, 10**9))
    selected.append(picked)
    used_videos |= set(picked.video_id)

sel = pd.concat(selected) if selected else pd.DataFrame(columns=neg_pool.columns)
print(f"selected negatives: {len(sel)} (target ~= {sum(pos_cell_target.values())}, " f"videos used: {sel.video_id.nunique()})")

results = []

for _, row in df[df.label == 1].iterrows():
    if pd.isna(row.time_of_event):
        continue
    vid = str(row.video_id)
    clip_path = OUTPUT_DIR / f"{vid}.npy"
    d = get_duration(row.video_file)
    if d is None: continue
    ev = float(row.time_of_event)
    st, en = ev - WARNING_SECONDS - CLIP_SECONDS, ev - WARNING_SECONDS
    if st < 0 or en > d: continue
    if clip_path.exists():
        results.append(dict(video_id=vid, label=1, status="ok_existing", clip_path=str(clip_path), video_duration=d, start_time=st, end_time=en , time_after_clip=d-en , padded_seconds=0, reason=None))
        continue
    clip, info = extract_pre_crash_clip(row.video_file, ev, clip_path, CLIP_SECONDS, WARNING_SECONDS, NUM_FRAMES, IMAGE_SIZE, short_clip="skip")
    results.append(dict(video_id=vid, label=1, status=info["status"],
                        clip_path=str(clip_path) if clip is not None else None,
                        video_duration=d, start_time=info.get("start_time"),
                        end_time=info.get("end_time"),
                        time_after_clip=d - info["end_time"] if info.get("end_time") else None,
                        padded_seconds=info.get("padded_seconds", 0),
                        reason=info.get("reason")))

for r in tqdm(sel.itertuples(), total=len(sel), desc="extract selected neg"):
    clip_path = OUTPUT_DIR / f"{r.video_id}.npy"
    if clip_path.exists(): clip_path.unlink()
    clip, info = extract_pre_crash_clip(r.video_id and Path(str(df.loc[r.row_idx, "video_file"])),
                                        r.target_time, clip_path,
                                        CLIP_SECONDS, WARNING_SECONDS,
                                        NUM_FRAMES, IMAGE_SIZE, short_clip="skip")
    results.append(dict(video_id=r.video_id, label=0, status=info["status"],
                        clip_path=str(clip_path) if clip is not None else None,
                        video_duration=r.duration,
                        start_time=info.get("start_time"), end_time=info.get("end_time"),
                        time_after_clip=(r.duration - info["end_time"]) if info.get("end_time") else None,
                        padded_seconds=info.get("padded_seconds", 0),
                        reason=info.get("reason")))

proc = pd.DataFrame(results)
proc.to_csv(OUTPUT_DIR / "processing_log.csv", index=False)
print("\nstatus:\n", proc.status.value_counts(dropna=False))
print("\nby class:\n", proc[proc.status.isin(["ok","ok_existing"])].groupby("label").size())


m = proc[proc.status.isin(["ok","ok_existing"])].dropna(subset=["start_time","time_after_clip","video_duration"]).copy()
if m.label.nunique() == 2 and len(m) > 20:
    P, N = m[m.label==1], m[m.label==0]
    print("\nKS start_time      :", ks_2samp(P.start_time, N.start_time))
    print("KS time_after_clip :", ks_2samp(P.time_after_clip, N.time_after_clip))
    print("KS video_duration  :", ks_2samp(P.video_duration, N.video_duration))

    def fitted_auc(feats):
        X = m[feats].astype(float).values; y = m.label.values
        skf = StratifiedKFold(5, shuffle=True, random_state=42)
        aucs = []
        for tr, te in skf.split(X, y):
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
            clf.fit(X[tr], y[tr])
            aucs.append(roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1]))
        return float(np.mean(aucs)), float(np.std(aucs))

    for name, feats in [("start_time only", ["start_time"]),
                        ("time_after_clip only", ["time_after_clip"]),
                        ("video_duration only", ["video_duration"]),
                        ("timing only", ["start_time", "time_after_clip"]),
                        ("timing + duration", ["start_time", "time_after_clip", "video_duration"])]:
        a, s = fitted_auc(feats)
        print(f"AUC {name:22s}: {a:.3f} +/- {s:.3f}")
else:
    print("not enough valid clips to verify")

NameError: name 'pos_start_times' is not defined

In [ ]:


OUTPUT_DIR = Path("nexar_data/processed_frames_matched2")

proc = pd.read_csv(OUTPUT_DIR / "processing_log.csv")
df = pd.read_csv("nexar_data/metadata.csv")
df["label"] = df["time_of_event"].notna().astype("int64")

# Sampling weights
ALERT_WEIGHT = 10.0
POSITIVE_OUTSIDE_ALERT_WEIGHT = 2.0
NEGATIVE_WEIGHT = 1.0

manifest = proc[proc["status"].isin(["ok", "ok_existing"])].copy()
metadata = df[["video_id", "time_of_alert", "time_of_event"]].drop_duplicates("video_id").copy()
manifest["video_id"] = manifest["video_id"].astype(str)
metadata["video_id"] = metadata["video_id"].astype(str)

manifest = manifest.merge(metadata, on="video_id", how="left", validate="many_to_one")

for col in ["time_of_alert", "time_of_event", "end_time"]:
    manifest[col] = pd.to_numeric(manifest[col], errors="coerce")

manifest["anchor_time"] = manifest["end_time"]

manifest["target_label"] = np.nan
manifest["sample_weight"] = np.nan
manifest["temporal_group"] = "invalid"

negative = manifest["label"] == 0

manifest.loc[negative, "target_label"] = 0
manifest.loc[negative, "sample_weight"] = NEGATIVE_WEIGHT
manifest.loc[negative, "temporal_group"] = "negative"

valid_positive = (
    (manifest["label"] == 1)
    & manifest["time_of_alert"].notna()
    & manifest["time_of_event"].notna()
    & (manifest["time_of_alert"] < manifest["time_of_event"])
)

alert_active = (
    valid_positive
    & (manifest["anchor_time"] >= manifest["time_of_alert"])
    & (manifest["anchor_time"] < manifest["time_of_event"])
)

manifest.loc[alert_active, "target_label"] = 1
manifest.loc[alert_active, "sample_weight"] = ALERT_WEIGHT
manifest.loc[alert_active, "temporal_group"] = "alert_to_event"

before_alert = (valid_positive & (manifest["anchor_time"] < manifest["time_of_alert"]))
manifest.loc[before_alert, "target_label"] = 0
manifest.loc[before_alert, "sample_weight"] = POSITIVE_OUTSIDE_ALERT_WEIGHT
manifest.loc[before_alert, "temporal_group"] = "positive_before_alert"


manifest.to_csv(OUTPUT_DIR / "temporal_training_manifest.csv",index=False)


## looggsssss
print("Original extracted clips:", len(proc))
print("Usable extracted clips:", len(manifest))
print("\nTemporal groups:")
print(manifest["temporal_group"].value_counts())
print("\nTarget labels:")
print(manifest["target_label"].value_counts(dropna=False))
print("\nSampling weights:")
print(manifest.groupby("temporal_group")["sample_weight"].first())
print("\nPositive clips excluded due to missing/invalid annotations:")
print(manifest[(manifest["label"] == 1) & manifest["target_label"].isna()].shape[0])
print("\nSaved temporal manifest successfully.")


Original extracted clips: 1364
Usable extracted clips: 1364

Temporal groups:
temporal_group
negative                 616
alert_to_event           560
positive_before_alert    188
Name: count, dtype: int64

Target labels:
target_label
0.0    804
1.0    560
Name: count, dtype: int64

Sampling weights:
temporal_group
alert_to_event           10.0
negative                  1.0
positive_before_alert     2.0
Name: sample_weight, dtype: float64

Positive clips excluded due to missing/invalid annotations:
0

Saved temporal manifest successfully.


In [ ]:


train_data = manifest.dropna(
    subset=["target_label", "clip_path", "sample_weight"]
).copy()

train_data["target_label"] = train_data["target_label"].astype(int)
train_data["video_id"] = train_data["video_id"].astype(str)


sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf.split(X=train_data, y=train_data["target_label"], groups=train_data["video_id"]))


train_df = train_data.iloc[train_idx].copy()
val_df = train_data.iloc[val_idx].copy()

train_df.to_csv(OUTPUT_DIR / "train_manifest.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val_manifest.csv", index=False)

# checking for video leakage
overlap = set(train_df["video_id"]) & set(val_df["video_id"])




#### logsssss
print("Training clips:", len(train_df))
print("Validation clips:", len(val_df))
print("\nTraining videos:", train_df["video_id"].nunique())
print("Validation videos:", val_df["video_id"].nunique())
print("\nTraining labels:")
print(train_df["target_label"].value_counts())
print("\nValidation labels:")
print(val_df["target_label"].value_counts())
print("\nVideo overlap:", len(overlap))
assert len(overlap) == 0, "Video leakage detected!"


Training clips: 1091
Validation clips: 273

Training videos: 1050
Validation videos: 262

Training labels:
target_label
0    643
1    448
Name: count, dtype: int64

Validation labels:
target_label
0    161
1    112
Name: count, dtype: int64

Video overlap: 0
